# A3.6 · Egress control for agents

**Function A — Security Architecture & Platform → The Platform & Cloud Security Engineer**  ·  *Security of AI*

Builds on **[A3.5 · MCP is not a security boundary](https://spbreed.github.io/cyber-commons/lessons/A3.5.html)**.

| | |
|---|---|
| Open-source tooling | Squid, Cilium, Kyverno |
| Open-weight models | GLM-4.6 |

> **Runs anywhere.** Every line of code is in this notebook — nothing to install, nothing to clone, no API key, no network. Standard library only, so it works on a Kaggle kernel with the internet switched off.

## 1 · The concept


Egress control is the difference between a compromised agent and a data breach.

The rule is simple and almost never followed: **allowlist by host, deny
everything else.** The two ways it goes wrong are both attempts to be helpful.

**Suffix allowlists.** `*.com` or `*.amazonaws.com` looks like a reasonable
scoping and permits exfiltration to any host in a namespace you do not control.

**Forgetting link-local.** `169.254.169.254` is the cloud instance metadata
service. Reaching it from a compromised workload yields the instance's IAM role
credentials — which is the pivot in a large fraction of real cloud breaches.
IMDSv2 makes it harder by requiring a PUT to get a token first, but an agent
that can make arbitrary HTTP requests can do a PUT.

There is also a failure that URL-level allowlisting cannot catch at all: **DNS
rebinding**, where an allowlisted hostname resolves to an internal address. That
one has to be handled at the connection layer, and this lesson shows why rather
than asserting it.

## 2 · Demo — a correct allowlist, and the destinations it refuses

In [ ]:
import re, socket
from urllib.parse import urlparse
from dataclasses import dataclass, field

PRIVATE_NETS = [
    ("127.0.0.0/8",     lambda o: o[0] == 127,                      "loopback"),
    ("10.0.0.0/8",      lambda o: o[0] == 10,                       "RFC1918 private"),
    ("172.16.0.0/12",   lambda o: o[0] == 172 and 16 <= o[1] <= 31, "RFC1918 private"),
    ("192.168.0.0/16",  lambda o: o[0] == 192 and o[1] == 168,      "RFC1918 private"),
    ("169.254.0.0/16",  lambda o: o[0] == 169 and o[1] == 254,      "link-local / cloud metadata"),
]

def classify_ip(ip):
    try:
        octets = [int(x) for x in ip.split(".")]
        if len(octets) != 4: return None
    except ValueError:
        return None
    for cidr, test, label in PRIVATE_NETS:
        if test(octets): return f"{label} ({cidr})"
    return None

@dataclass
class EgressPolicy:
    allow_hosts: set = field(default_factory=set)
    resolver: dict = field(default_factory=dict)   # host -> ip, stands in for DNS

    def check(self, url):
        host = (urlparse(url).hostname or "").lower()
        if not host: return False, "unparseable destination"
        # literal IP in the URL
        cls = classify_ip(host)
        if cls: return False, f"blocked: {cls}"
        if host not in self.allow_hosts:
            return False, "not on the egress allowlist (deny by default)"
        return True, "host allowlisted"

pol = EgressPolicy(allow_hosts={"api.github.com", "pypi.org"})
URLS = [
 "https://api.github.com/repos/x/y",
 "https://pypi.org/simple/requests/",
 "http://169.254.169.254/latest/meta-data/iam/security-credentials/",
 "http://127.0.0.1:8080/admin",
 "http://10.0.3.14:9200/_search",
 "https://collect.example.com/upload",
 "https://pastebin.com/api/api_post.php",
]
for u in URLS:
    ok, why = pol.check(u)
    print(f"{'ALLOW' if ok else 'DENY ':5s} {u[:58]:60s} {why}")

## 3 · Where it breaks — the helpful allowlist

This configuration passes a config review. Someone wrote it because the agent legitimately needed several AWS endpoints and listing them all was tedious.

In [ ]:
@dataclass
class SuffixPolicy:
    allow_suffixes: set
    def check(self, url):
        host = (urlparse(url).hostname or "").lower()
        if not host: return False, "unparseable"
        # Iterating the set directly would report whichever suffix Python
        # happened to visit first — and set order depends on PYTHONHASHSEED,
        # so the audit line would name a different rule on a different
        # machine. Report the most specific match, the way a router reports
        # the longest matching prefix: the answer to "which rule let this
        # through" has to be reproducible or it is not evidence.
        for suf in sorted(self.allow_suffixes, key=len, reverse=True):
            if host.endswith(suf):
                return True, f"matches suffix {suf}"
        return False, "no suffix match"

loose = SuffixPolicy(allow_suffixes={".com", ".amazonaws.com"})
print("suffix allowlist — looks tidy, permits the internet:")
for u in URLS:
    ok, why = loose.check(u)
    flag = "  ← EXFILTRATION PATH" if ok and "collect" in u or ok and "pastebin" in u else ""
    print(f"{'ALLOW' if ok else 'DENY ':5s} {u[:58]:60s} {why}{flag}")

attacker_bucket = "https://attacker-controlled.s3.amazonaws.com/loot"
ok, why = loose.check(attacker_bucket)
print(f"\n{'ALLOW' if ok else 'DENY '} {attacker_bucket}  {why}")
print("An attacker's own S3 bucket matches '.amazonaws.com'. The suffix that")
print("was added to reduce toil is now the exfiltration channel.")

## 4 · The harder failure — DNS rebinding

The URL is on the allowlist. The hostname resolves to the metadata service. URL-level checking is structurally unable to catch this, because the decision is made before the name is resolved — and the name can resolve differently on the next lookup.

In [ ]:
# the attacker controls DNS for a hostname you allowlisted for a legitimate reason
DNS = {
    "api.github.com":        "140.82.121.5",
    "pypi.org":              "151.101.0.223",
    "metrics.partner.example": "169.254.169.254",     # attacker-controlled record
}
pol2 = EgressPolicy(allow_hosts={"api.github.com", "pypi.org", "metrics.partner.example"},
                    resolver=DNS)

url = "https://metrics.partner.example/collect"
ok, why = pol2.check(url)
print(f"URL-level check:      {'ALLOW' if ok else 'DENY '} {why}")
print(f"but it resolves to:   {DNS['metrics.partner.example']} "
      f"({classify_ip(DNS['metrics.partner.example'])})")

def check_after_resolution(policy, url):
    """The control has to run on the RESOLVED ADDRESS, at connect time."""
    ok, why = policy.check(url)
    if not ok: return False, why
    host = urlparse(url).hostname.lower()
    ip = policy.resolver.get(host)
    if ip is None: return False, "unresolvable"
    cls = classify_ip(ip)
    if cls: return False, f"resolved address is {cls} — refusing ({host} → {ip})"
    return True, f"allowlisted and resolves to public {ip}"

print("\nsame URLs, checked after resolution:")
for u in ["https://api.github.com/x", "https://metrics.partner.example/collect"]:
    ok, why = check_after_resolution(pol2, u)
    print(f"   {'ALLOW' if ok else 'DENY ':5s} {u[:44]:46s} {why}")

In [ ]:
# Verify: score the three policies against the full URL set + the rebind.
CASES = [(u, False) for u in URLS[2:]] + [(URLS[0], True), (URLS[1], True),
                                          ("https://metrics.partner.example/c", False)]
def score(name, check):
    wrong = []
    for url, should_allow in CASES:
        got = check(url)
        if got != should_allow: wrong.append(url)
    print(f"{name:34s} incorrect decisions: {len(wrong)}")
    for w in wrong: print(f"      {w[:66]}")

pol3 = EgressPolicy(allow_hosts={"api.github.com", "pypi.org",
                                 "metrics.partner.example"}, resolver=DNS)
score("suffix allowlist",        lambda u: loose.check(u)[0])
score("host allowlist (URL only)", lambda u: pol3.check(u)[0])
score("host allowlist + resolution", lambda u: check_after_resolution(pol3, u)[0])

## What you just proved

The host allowlist permits the two legitimate destinations and denies the metadata service, the loopback and private addresses, and both exfiltration hosts. The suffix allowlist permits both exfiltration paths plus an attacker's S3 bucket. The rebinding case passes the URL-level check and is only caught once the resolved address is classified — the resolution-aware policy is the only one with zero incorrect decisions.

## Your turn

Check whether your agent's egress control runs before or after DNS resolution. If it is a URL allowlist in application code, it is before, and the rebinding case is open. Moving the check to the proxy or the CNI is the fix.

---

**Next → [A3.7 · Runtime containment levers](https://spbreed.github.io/cyber-commons/lessons/A3.7.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/A3.6.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/A3.6.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*